In [1]:
import os
import warnings
warnings.filterwarnings('ignore')
warnings.simplefilter(action='ignore', category=FutureWarning)

import numpy as np
import pyarrow
import polars as pl

import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (log_loss, accuracy_score, classification_report,
                             roc_curve, roc_auc_score, RocCurveDisplay,
                             ConfusionMatrixDisplay, confusion_matrix,
                             precision_score, recall_score, f1_score)

import xgboost as xgb
from xgboost import XGBClassifier, plot_importance

from scipy.stats import randint, uniform

In [2]:

import yfinance as yf

start_date = '2010-01-01'
end_date   = '2026-04-16'

for sym, fname in [('AAPL', 'aapl_historical_data.csv'),
                   ('SPY',  'spy_historical_data.csv'),
                   ('QQQ',  'qqq_historical_data.csv'),
                   ('^VIX', 'vix_historical_data.csv'),
                   ('^TNX', 'tnx_historical_data.csv')]:
    print(f'pulling {sym}...')
    t = yf.Ticker(sym)
    h = t.history(start=start_date, end=end_date, auto_adjust=True)
    print(f'  rows: {len(h)}')
    h.to_csv(fname)

print('done')

pulling AAPL...
  rows: 4095
pulling SPY...
  rows: 4095
pulling QQQ...
  rows: 4095
pulling ^VIX...
  rows: 4095
pulling ^TNX...
  rows: 4093
done


In [3]:
aapl = pl.read_csv('aapl_historical_data.csv', try_parse_dates=True)

# drop junk columns
drop_cols = [c for c in ['Dividends','Stock Splits','StockSplits','Capital Gains']
             if c in aapl.columns]
aapl = aapl.drop(drop_cols)

# --- core return (target source) ---
aapl = aapl.with_columns(
    np.log(pl.col('Close') / pl.col('Close').shift()).alias('LogReturn')
)

# --- return lags (replaces CloseLag1/2/3) ---
aapl = aapl.with_columns(pl.col('LogReturn').shift(1).alias('RetLag1'))
aapl = aapl.with_columns(pl.col('LogReturn').shift(2).alias('RetLag2'))
aapl = aapl.with_columns(pl.col('LogReturn').shift(3).alias('RetLag3'))

# --- percentage HML and OMC, plus lags ---
aapl = aapl.with_columns(((pl.col('High') - pl.col('Low')) / pl.col('Close')).alias('HMLpct'))
aapl = aapl.with_columns(((pl.col('Open') - pl.col('Close')) / pl.col('Close')).alias('OMCpct'))

aapl = aapl.with_columns(pl.col('HMLpct').shift(1).alias('HMLpctLag1'))
aapl = aapl.with_columns(pl.col('HMLpct').shift(2).alias('HMLpctLag2'))
aapl = aapl.with_columns(pl.col('HMLpct').shift(3).alias('HMLpctLag3'))

aapl = aapl.with_columns(pl.col('OMCpct').shift(1).alias('OMCpctLag1'))
aapl = aapl.with_columns(pl.col('OMCpct').shift(2).alias('OMCpctLag2'))
aapl = aapl.with_columns(pl.col('OMCpct').shift(3).alias('OMCpctLag3'))

# --- volume ratio vs 20-day average (stationary) ---
# use shifted volume so today's volume is never in a feature
aapl = aapl.with_columns(pl.col('Volume').shift(1).alias('Vol_t1'))
aapl = aapl.with_columns(
    pl.col('Vol_t1').rolling_mean(window_size=20).alias('Vol20')
)
aapl = aapl.with_columns((pl.col('Vol_t1') / pl.col('Vol20')).alias('VolRatioLag1'))
aapl = aapl.with_columns(pl.col('VolRatioLag1').shift(1).alias('VolRatioLag2'))
aapl = aapl.with_columns(pl.col('VolRatioLag1').shift(2).alias('VolRatioLag3'))

# --- return EMAs (replaces price EMAs) ---
# compute on RetLag1 so no same-day leakage
aapl = aapl.with_columns(pl.col('RetLag1').ewm_mean(half_life=1, ignore_nulls=True).alias('RetEMA2'))
aapl = aapl.with_columns(pl.col('RetLag1').ewm_mean(half_life=2, ignore_nulls=True).alias('RetEMA4'))
aapl = aapl.with_columns(pl.col('RetLag1').ewm_mean(half_life=4, ignore_nulls=True).alias('RetEMA8'))

# --- rolling realized vol (new, stationary) ---
aapl = aapl.with_columns(
    pl.col('RetLag1').rolling_std(window_size=20).alias('RetVol20')
)

# --- binary target ---
aapl = aapl.with_columns(
    pl.when(pl.col('LogReturn') > 0.0).then(pl.lit(1)).otherwise(pl.lit(0)).alias('Target')
)

print('aapl columns after v2 feature engineering:', aapl.columns)

aapl columns after v2 feature engineering: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'LogReturn', 'RetLag1', 'RetLag2', 'RetLag3', 'HMLpct', 'OMCpct', 'HMLpctLag1', 'HMLpctLag2', 'HMLpctLag3', 'OMCpctLag1', 'OMCpctLag2', 'OMCpctLag3', 'Vol_t1', 'Vol20', 'VolRatioLag1', 'VolRatioLag2', 'VolRatioLag3', 'RetEMA2', 'RetEMA4', 'RetEMA8', 'RetVol20', 'Target']


In [4]:
def load_and_prep(path, prefix):
    df = pl.read_csv(path, try_parse_dates=True)
    df = df.select(['Date', 'Close'])
    df = df.rename({'Close': f'{prefix}_Close'})
    df = df.with_columns(
        np.log(pl.col(f'{prefix}_Close') / pl.col(f'{prefix}_Close').shift()).alias(f'{prefix}_Ret')
    )
    df = df.with_columns(
        (pl.col(f'{prefix}_Close') - pl.col(f'{prefix}_Close').shift()).alias(f'{prefix}_Chg')
    )
    # lag everything by one day
    df = df.with_columns(pl.col(f'{prefix}_Ret').shift(1).alias(f'{prefix}_RetLag1'))
    df = df.with_columns(pl.col(f'{prefix}_Close').shift(1).alias(f'{prefix}_LvlLag1'))
    df = df.with_columns(pl.col(f'{prefix}_Chg').shift(1).alias(f'{prefix}_ChgLag1'))
    return df.select(['Date', f'{prefix}_RetLag1', f'{prefix}_LvlLag1', f'{prefix}_ChgLag1'])

spy = load_and_prep('spy_historical_data.csv', 'SPY')
qqq = load_and_prep('qqq_historical_data.csv', 'QQQ')
vix = load_and_prep('vix_historical_data.csv', 'VIX')
tnx = load_and_prep('tnx_historical_data.csv', 'TNX')

# for the final feature set we only keep the most informative signal per series
# (keep return for SPY/QQQ, level+change for VIX and TNX)
spy = spy.select(['Date', 'SPY_RetLag1'])
qqq = qqq.select(['Date', 'QQQ_RetLag1'])
vix = vix.select(['Date', 'VIX_LvlLag1', 'VIX_ChgLag1'])
tnx = tnx.select(['Date', 'TNX_ChgLag1'])

# join on Date
aapl = aapl.join(spy, on='Date', how='left')
aapl = aapl.join(qqq, on='Date', how='left')
aapl = aapl.join(vix, on='Date', how='left')
aapl = aapl.join(tnx, on='Date', how='left')

# drop rows with any nulls (from lags, rolling windows, and any missing exogenous dates)
aapl = aapl.drop_nulls()

print('final row count:', aapl.height)
print('date range:', aapl['Date'].min(), 'to', aapl['Date'].max())
print()
print('all columns:', aapl.columns)

final row count: 0
date range: None to None

all columns: ['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'LogReturn', 'RetLag1', 'RetLag2', 'RetLag3', 'HMLpct', 'OMCpct', 'HMLpctLag1', 'HMLpctLag2', 'HMLpctLag3', 'OMCpctLag1', 'OMCpctLag2', 'OMCpctLag3', 'Vol_t1', 'Vol20', 'VolRatioLag1', 'VolRatioLag2', 'VolRatioLag3', 'RetEMA2', 'RetEMA4', 'RetEMA8', 'RetVol20', 'Target', 'SPY_RetLag1', 'QQQ_RetLag1', 'VIX_LvlLag1', 'VIX_ChgLag1', 'TNX_ChgLag1']


In [5]:
def load_and_prep(path, prefix):
    """Load an exogenous series, compute log return and level change, lag by 1 day."""
    df = pl.read_csv(path, try_parse_dates=True)
    df = df.select(['Date', 'Close'])
    # normalize Date to a plain date (strips timezone and time-of-day)
    df = df.with_columns(pl.col('Date').cast(pl.Date))
    df = df.rename({'Close': f'{prefix}_Close'})
    df = df.with_columns(
        np.log(pl.col(f'{prefix}_Close') / pl.col(f'{prefix}_Close').shift()).alias(f'{prefix}_Ret')
    )
    df = df.with_columns(
        (pl.col(f'{prefix}_Close') - pl.col(f'{prefix}_Close').shift()).alias(f'{prefix}_Chg')
    )
    df = df.with_columns(pl.col(f'{prefix}_Ret').shift(1).alias(f'{prefix}_RetLag1'))
    df = df.with_columns(pl.col(f'{prefix}_Close').shift(1).alias(f'{prefix}_LvlLag1'))
    df = df.with_columns(pl.col(f'{prefix}_Chg').shift(1).alias(f'{prefix}_ChgLag1'))
    return df.select(['Date', f'{prefix}_RetLag1', f'{prefix}_LvlLag1', f'{prefix}_ChgLag1'])

# normalize aapl Date the same way before joining
aapl = aapl.with_columns(pl.col('Date').cast(pl.Date))

spy = load_and_prep('spy_historical_data.csv', 'SPY')
qqq = load_and_prep('qqq_historical_data.csv', 'QQQ')
vix = load_and_prep('vix_historical_data.csv', 'VIX')
tnx = load_and_prep('tnx_historical_data.csv', 'TNX')

spy = spy.select(['Date', 'SPY_RetLag1'])
qqq = qqq.select(['Date', 'QQQ_RetLag1'])
vix = vix.select(['Date', 'VIX_LvlLag1', 'VIX_ChgLag1'])
tnx = tnx.select(['Date', 'TNX_ChgLag1'])

aapl = aapl.join(spy, on='Date', how='left')
aapl = aapl.join(qqq, on='Date', how='left')
aapl = aapl.join(vix, on='Date', how='left')
aapl = aapl.join(tnx, on='Date', how='left')

# check null counts before dropping so we can see what's going on
print('null counts per column before drop_nulls:')
for col in aapl.columns:
    n = aapl[col].null_count()
    if n > 0:
        print(f'  {col}: {n}')
print(f'total rows before drop_nulls: {aapl.height}')

aapl = aapl.drop_nulls()

print()
print(f'final row count: {aapl.height}')
print(f'date range: {aapl["Date"].min()} to {aapl["Date"].max()}')

null counts per column before drop_nulls:
total rows before drop_nulls: 0

final row count: 0
date range: None to None


In [7]:
print('n_total:  ', aapl.height, type(aapl.height))
print('expected n_holdout:', int(0.15 * aapl.height))

n_total:   0 <class 'int'>
expected n_holdout: 0
